In [32]:
import pandas as pd
import requests

ndc = '00378581577'
url = f"http://localhost:4000/REST/ndcproperties.json?id={ndc}"

response = requests.get(url)

if response.status_code == 200:
    json_data = response.json()
    
    # Extract the array from the wrapper
    ndc_properties = json_data['ndcPropertyList']['ndcProperty']
    
    # Option 1: Basic flattening
    # df = pd.json_normalize(ndc_properties)
    # display(df.head())
    
    # Option 2: Flatten the properties fully (RECOMMENDED)
    df_properties = pd.json_normalize(
        ndc_properties,
        record_path=['propertyConceptList', 'propertyConcept'],
        meta=['ndcItem', 'ndc9', 'ndc10', 'rxcui', 'source', 'splSetIdItem'],
        errors='ignore'
    )
    
    # Create a wide format for easier access
    df_wide = df_properties.pivot_table(
        index=['ndcItem', 'rxcui', 'source'],
        columns='propName',
        values='propValue',
        aggfunc='first'
    ).reset_index()
    
    display(df_wide)
    
else:
    print(f"Error: Status code {response.status_code}")

propName,ndcItem,rxcui,source,ANDA,COLOR,COLORTEXT,IMPRINT_CODE,LABELER,LABEL_TYPE,MARKETING_CATEGORY,MARKETING_EFFECTIVE_TIME_LOW,MARKETING_STATUS,SCORE,SHAPE,SIZE
0,00378581577,349200,Hybrid,ANDA090866,C48327,PURPLE(dark violet),M;V15,Mylan Pharmaceuticals Inc.,HUMAN PRESCRIPTION DRUG,ANDA,20150105,ACTIVE,1,C48345,21 mm


In [33]:
import pandas as pd
import requests

ndc = '00378581577'
url = f"http://localhost:4000/REST/ndcproperties.json?id={ndc}"

response = requests.get(url)


if response.status_code == 200:
    json_data = response.json()
    ndc_properties = json_data['ndcPropertyList']['ndcProperty']
    
    # Normalize just the main level first
    df_base = pd.json_normalize(ndc_properties)
    
    # Extract packaging
    df_base['packaging'] = df_base['packagingList.packaging'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
    )
    
    # Now normalize properties
    df_properties = pd.json_normalize(
        ndc_properties,
        record_path=['propertyConceptList', 'propertyConcept'],
        meta=['ndcItem', 'rxcui', 'source'],
        errors='ignore'
    )
    
    # Pivot
    df_wide = df_properties.pivot_table(
        index=['ndcItem', 'rxcui', 'source'],
        columns='propName',
        values='propValue',
        aggfunc='first'
    ).reset_index()
    
    # Merge with packaging
    df_final = df_wide.merge(
        df_base[['ndcItem', 'rxcui', 'source', 'packaging']], 
        on=['ndcItem', 'rxcui', 'source']
    )
    
    display(df_final)

,ndcItem,rxcui,source,ANDA,COLOR,COLORTEXT,IMPRINT_CODE,LABELER,LABEL_TYPE,MARKETING_CATEGORY,MARKETING_EFFECTIVE_TIME_LOW,MARKETING_STATUS,SCORE,SHAPE,SIZE,packaging
0,00378581577,349200,Hybrid,ANDA090866,C48327,PURPLE(dark violet),M;V15,Mylan Pharmaceuticals Inc.,HUMAN PRESCRIPTION DRUG,ANDA,20150105,ACTIVE,1,C48345,21 mm,"90 TABLET, FILM COATED in 1 BOTTLE, PLASTIC (0..."
